Code for generating 2 agent simulations where goal directions are orthogonal (figure 11[right]); with PBA

setup periodic in x and y, but y limits are very large so it's practically unbounded in y

# PBA

In [ ]:
#imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist
import matplotlib.patches as patches
from math import cos, sin, pi
from parfor import parfor
import csv
from tqdm import tqdm
import pickle

In [1]:
#Functions for SFM

def selfPropulsionForce(m, tau, desiredSpeed, desiredDirection, velocity):
    desiredVelocity = desiredSpeed * desiredDirection
    return (desiredVelocity - velocity) * (m / tau)

def interactionForce(position, R, A, B, Lx, Ly):
    N = position.shape[0]
    epsilon = 1e-6 # Used to avoid infinity
    agent_size = 2 * R

    x = position[:, 0].reshape(-1, 1)
    y = position[:, 1].reshape(-1, 1)

    dx = x - x.T
    dy = y - y.T


    dx_u = dx - np.round(dx / Lx) * Lx
    dy_u = dy - np.round(dy / Ly) * Ly

    # Euclidean distance calculation
    dist = np.sqrt(dx_u**2 + dy_u**2)
    
    valid_index = (dist < (3*R)) & (dist > 0)

    # Unit vectors
    dij_x = np.zeros_like(dist)
    dij_y = np.zeros_like(dist)
    np.divide(dx_u, dist, out=dij_x, where=dist>epsilon)
    np.divide(dy_u, dist, out=dij_y, where=dist>epsilon)   
    
    # Apply valid_index mask
    dij_x[~valid_index] = 0
    dij_y[~valid_index] = 0
    
    # Effective distance
    effective_dist = np.maximum(dist - agent_size, epsilon)
    
    # Force magnitude calculation
    force_magnitude = np.zeros_like(dist)
    np.power(effective_dist, -B, out=force_magnitude, where=effective_dist>epsilon)
    force_magnitude *= -A
    
    force_magnitude[~valid_index] = 0

    interaction_force = np.column_stack((np.sum(force_magnitude * dij_x, axis=0),np.sum(force_magnitude * dij_y, axis=0)))

    return interaction_force


In [2]:
#accounting for boundary conditions
def periodify(pos,Lx,Ly):
    if len(pos.shape)==1:
        pos[0]=pos[0]%Lx
        pos[1]=pos[1]%Ly
    else:
        pos[:,0]=pos[:,0]%Lx
        pos[:,1]=pos[:,1]%Ly
    return pos



def getdisp(a,b):
    disp=a-b
    if len(disp.shape)==1:
        disp[0]=disp[0]-np.round(disp[0]/Lx)*Lx
        disp[1]=disp[1]-np.round(disp[1]/Ly)*Ly
    else:
        disp[:,0]=disp[:,0]-np.round(disp[:,0]/Lx)*Lx
        disp[:,1]=disp[:,1]-np.round(disp[:,1]/Ly)*Ly
    return disp





In [1]:
#Functions for PBA; Uncomment lines in make_moveset and randevolve to remove restriction on moves

def updatepos(agentpos,agentdir,v):
    return periodify(agentpos+np.array([v*np.cos(agentdir),v*np.sin(agentdir)]),Lx,Ly)

def getstate(agentposdir, agent, othersposes,grad):
    return tuple([np.round(agentposdir[0][0],6),np.round(agentposdir[0][1],6)]+[np.round(np.exp(grad) if grad<0 else 1+np.log(1+grad),6)])


def getbest(stateslists):
    outs=[]
    for statelist in stateslists:
        statematrix=np.array(statelist)
        if statematrix.shape[0]==0:
            score=-1
        else:
            gradmat=statematrix[:,-1]
            score=gradmat.sum()
        outs.append(score)
    return randargmax(np.array(outs))
    
def randargmax(b):
    return np.random.choice(np.where(b == b.max())[0])


def findposesovertime(agentsposes,agentsdirs,agentsvels,tau_fut):
    posesovertime=[]
    posesovertime.append(agentsposes)
    newposes=agentsposes.copy()
    newdirs=agentsdirs.copy()
    for i in range(tau_fut):
        newposes=newposes+agentsvels.reshape(-1,1)*np.vstack([np.cos(newdirs),np.sin(newdirs)]).T
        newpos,newdirs=(newposes.copy(),newdirs)
        posesovertime.append(periodify(newpos,Lx,Ly))
    return posesovertime


def make_moveset(agentpos,agentdir,v):
    dircombs=[(agentdir+deltheta)%(2*np.pi),agentdir,(agentdir-deltheta)%(2*np.pi),(agentdir-2*deltheta)%(2*np.pi),(agentdir+2*deltheta)%(2*np.pi)]
    # dircombs=[(agentdir+deltheta)%(2*np.pi),agentdir,(agentdir-deltheta)%(2*np.pi)]
    velcombs=[v]
    moveset=[]
    for vel in velcombs:
        for dir in dircombs:
            moveset.append([vel,dir])
    return moveset


def reflect_nearest(agentpos,agent,othersposes):
    disps=getdisp(othersposes,agentpos)
    disps=np.delete(disps,agent,axis=0)
    dists=((disps**2).sum(axis=1))**0.5
    minarg=dists.argmin()
    if dists[minarg]<2*R:
        mag=2*R-dists[minarg]
        dir=np.arctan2(-disps[minarg][1],-disps[minarg][0])
        newpos=agentpos+[2*mag*np.cos(dir),2*mag*np.sin(dir)]
        return (periodify(newpos,Lx,Ly),dir)[0]
    return agentpos
        
def randevolve(numsamp,agentpos,agentdir, agent, othersposesovertime, move, k,tau_fut,velprobs,thetprobs):
    v=1 
    outposes=[]
    ogpos=agentpos
    othersposes=othersposesovertime[1]
    newagentdir=(move[1])%(2*np.pi)
    newagentposdir=[reflect_nearest(updatepos(agentpos,newagentdir,v),agent,othersposes),newagentdir]
    for samp in range(numsamp):
        newpos,newdir=newagentposdir.copy()
        for t in range(2,tau_fut+1):
            newdir=newdir+np.random.choice([-deltheta,-2*deltheta,0,deltheta,2*deltheta])
            # newdir=np.random.choice([i for i in np.arange(0,2*np.pi-0.0001,np.pi/5)])
            newpos=reflect_nearest(updatepos(newpos,newdir,v),agent,othersposesovertime[t])
        outposes.append(getstate([newpos,newdir],agent,othersposesovertime[t],getdisp(newpos,ogpos)@np.array([np.cos(desired[agent]),np.sin(desired[agent])])))
    return outposes


# Parallel

In [5]:
from tqdm import tqdm
import pickle


deltaT = 0.005
timeStep = 35000
totalTime = deltaT*timeStep




N=2
Lx=20
Ly=500
R = 1
m = 1
A = 0.01
B = 3
tau = 1


updateT=tau
updateF=round(updateT/deltaT)



Nr = 1/2
deltheta=np.pi/8
numagents=N
tau_fut=6
agentradius=R

numsamp=50




g1 = int(round(N * Nr))
g2 = N - g1


In [6]:
for rep in range(15):
    xData = []
    yData = []
    uData = []
    vData = []
    desdirs=[]
    forces=[]

    position=np.array([[4,250],[16,250]])
    position=position+(np.random.random(size=position.shape)-0.5)*0.001
    position[:,0]=position[:,0]%Lx
    position[:,1]=position[:,1]%Ly
    
    speed=0.1

    theta = np.array([[np.pi/4],[3*np.pi/4]])    
    
    velocity = speed * np.column_stack((np.cos(theta), np.sin(theta)))
    
    desiredSpeed = np.ones((N,1)) * 1
    desiredDirectionA=theta.copy()
    desired=(desiredDirectionA.copy()).ravel()
    desiredDirection = np.hstack((np.cos(desiredDirectionA), np.sin(desiredDirectionA)))

    
    xData.append(position[:,0])
    yData.append(position[:,1])
    
    uData.append(velocity[:,0])
    vData.append(velocity[:,1])
    
    
    
    
    
    for t in tqdm(range(timeStep)):
        if t*deltaT>=0:#change to if t*deltaT<0 to stop updating desdir and run SFM without PBA
            if t%updateF==updateF-1:
                agentsposes=position
                agentsdirs=np.arctan2(velocity[:,1],velocity[:,0])%(2*np.pi)
                agentsvels=updateT*((velocity[:,1])**2+(velocity[:,0])**2)**0.5
                agentsposesovertime=findposesovertime(agentsposes,agentsdirs,agentsvels,tau_fut)
                chosenmoves=[]
                @parfor(range(numagents),bar=False)
                def paragent(focal):
                    stateslists=[]
                    agentpos=agentsposes[focal]
                    agentdir=agentsdirs[focal]
                    agentvel=agentsvels[focal]
                    moveset=make_moveset(agentpos,agentdir,agentvel)
                    for move in moveset:
                        evolvelist=randevolve(numsamp,agentpos,agentdir,focal,agentsposesovertime,move,1,tau_fut,[1/3]*3,[1/5]*5)
                        stateslists.append(evolvelist)
                    chosen=getbest(stateslists)
                    return moveset[chosen]
                chosenmoves=np.array(paragent)
                desiredDirectionA=(chosenmoves[:,1]).reshape(-1,1)
                desiredDirection = np.hstack((np.cos(desiredDirectionA), np.sin(desiredDirectionA)))
        desdirs.append(desiredDirection[:].copy())

        f1 = selfPropulsionForce(m, tau, desiredSpeed, desiredDirection, velocity)
        f2 = interactionForce(position, R, A, B, Lx,Ly)
        force = (f1+f2)/m

        newPosition = position + (velocity * deltaT) + (0.5 * force * deltaT**2)
        newPosition[:,0] = (newPosition[:,0]) % Lx
        newPosition[:,1] = (newPosition[:,1]) % Ly
    

        deltaV = force * deltaT
    
        vint = velocity + deltaV
    
        f1_new = selfPropulsionForce(m, tau, desiredSpeed, desiredDirection, vint) 
        f2_new =  interactionForce(newPosition, R, A, B, Lx,Ly)
        
        forceNew = (f1_new + f2_new)/m
        
        newVelocity = velocity + (0.5 * (force + forceNew)) * deltaT
    
        velocity = newVelocity
        position = newPosition
        

        xData.append(newPosition[:,0])
        yData.append(newPosition[:,1])
        
        uData.append(newVelocity[:,0])
        vData.append(newVelocity[:,1])
        forces.append([f1,f2])

    with open('./2agents_withfsm_ortho/2_agents_withfsm_ortho_'+str(rep)+'_'+str(np.random.randint(10000))+'.pickle','wb') as file:
        pickle.dump({'xData':xData,'yData':yData,'uData':uData,'vData':vData,'forces':forces,'desdirs':desdirs,'g1':g1,'Lx':Lx,'Ly':Ly, 'note':'ylim set very high,unbounded; agents have orthogonal desired directions'},file)#,'note':'tried to incorporate neighbor effects while predicting motion of other agents'},file)


    


100%|██████████████████████████████████████████████████████████████████████████| 35000/35000 [00:29<00:00, 1197.37it/s]
